In [ ]:
# In[1]:

import pandas as pd

# Load files
metric_fp = "metric.csv"
trace_fp = "trace.csv"
log_fp = "log.csv"
err_fp = "error_logs.csv"

df_metric = pd.read_csv(metric_fp)
df_trace = pd.read_csv(trace_fp)
df_log = pd.read_csv(log_fp)
df_err = pd.read_csv(err_fp)

# Parse timestamps to UTC datetime as required
for df in (df_metric, df_trace, df_log, df_err):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Ensure numeric 'value' where applicable
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Normalize KPI column names for grouping
df_metric = df_metric.rename(columns={'kpi_name': 'kpi_name'})
df_trace = df_trace.rename(columns={'trace_name': 'kpi_name'})
df_log = df_log.rename(columns={'log_name': 'kpi_name'})

# Aggregation function
def aggregate_kpis(df, source_label):
    # Group by component and KPI name, compute required stats on the full series
    g = df.groupby(['cmdb_id', 'kpi_name'])['value']
    agg = g.agg(
        count='count',
        min='min',
        p50=lambda x: x.quantile(0.5),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        max='max'
    ).reset_index()
    agg.insert(0, 'source', source_label)
    # Round numeric columns a bit for compactness
    for col in ['min', 'p50', 'p95', 'p99', 'max']:
        if col in agg.columns:
            agg[col] = pd.to_numeric(agg[col])
    return agg

agg_metric = aggregate_kpis(df_metric, 'metric')
agg_trace = aggregate_kpis(df_trace, 'trace')
agg_log = aggregate_kpis(df_log, 'log')

# Combine aggregated results
combined = pd.concat([agg_metric, agg_trace, agg_log], ignore_index=True)

# Sort by count descending and limit to at most 50 rows
combined_sorted = combined.sort_values('count', ascending=False).head(50)

# Short list of unique cmdb_id values across all files
all_cmdbs = pd.concat([
    df_metric['cmdb_id'].dropna(),
    df_trace['cmdb_id'].dropna(),
    df_log['cmdb_id'].dropna(),
    df_err['cmdb_id'].dropna()
]).unique()
unique_cmdbs = pd.Series(sorted(all_cmdbs), name='cmdb_id')

# Total number of rows in each file
counts_per_file = pd.Series({
    'metric.csv': df_metric.shape[0],
    'trace.csv': df_trace.shape[0],
    'log.csv': df_log.shape[0],
    'error_logs.csv': df_err.shape[0]
}).rename("row_count").reset_index().rename(columns={'index': 'file'})

# Display outputs (compact)
combined_sorted, unique_cmdbs, counts_per_file

```
Out[1]:
```
summary = (
    "Summary of aggregation results:\n"
    "- Files processed: metric.csv (1950 rows), trace.csv (0 rows), log.csv (450 rows), error_logs.csv (0 rows).\n"
    "- Unique components found (15): carts, carts-db, catalogue, catalogue-db, front-end, orders, orders-db, payment, "
    "queue-master, rabbitmq, rabbitmq-exporter, session-db, shipping, user, user-db.\n"
    "- Aggregation: for metric/trace/log we computed count, min, p50, p95, p99, max per (cmdb_id, KPI) using the entire series.\n"
    "- Key observations: the combined table (limited to 50 rows) is dominated by metric entries. Many metric component-KPIs have count=25 (e.g., carts: cpu, diskio, error, latency-50, latency-90).\n"
    "- Note: trace.csv and error_logs.csv contain no rows in this dataset, so no trace or raw error-log KPIs were aggregated.\n"
    "If you want more detail, I can show the top aggregated rows or the full combined table (up to 50 rows)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(    source       cmdb_id    kpi_name  count           min           p50           p95           p99           max
0   metric         carts         cpu     25  1.095733e+00  1.990448e+00  7.891882e+01  7.960786e+01  7.980687e+01
1   metric         carts      diskio     25  0.000000e+00  0.000000e+00  0.000000e+00  9.928308e+04  1.306356e+05
2   metric         carts       error     25  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
3   metric         carts  latency-50     25  1.774416e-02  1.857944e-02  2.237568e-02  2.311620e-02  2.334863e-02
4   metric         carts  latency-90     25  3.998200e-02  4.290086e-02  6.019077e-02  6.423516e-02  6.532552e-02
..     ...           ...         ...    ...           ...           ...           ...           ...           ...
45  metric       payment    workload     25  1.749750e+00  2.030167e+00  2.163553e+00  2.200469e+00  2.207617e+00
46  metric  queue-master         cpu     25  2.568197e-01  3.204087e-01  3.560962e-01  4.046984e-01  4.199352e-01
47  metric  queue-master      diskio     25  2.156452e+04  3.710150e+04  5.679773e+04  6.407248e+04  6.564979e+04
48  metric  queue-master         mem     25  2.700083e+08  2.700095e+08  2.701238e+08  2.701247e+08  2.701249e+08
49  metric  queue-master      socket     25  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00  3.000000e+00

[50 rows x 9 columns], 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object,              file  row_count
0      metric.csv       1950
1       trace.csv          0
2         log.csv        450
3  error_logs.csv          0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing DataFrames: df_metric, df_log
import pandas as pd
import numpy as np

# Incident window (UTC)
incident_start = pd.to_datetime('2024-01-19 15:42:56', utc=True)
incident_end   = pd.to_datetime('2024-01-19 16:12:56', utc=True)

# Ensure value columns numeric (they should already be)
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# 1) Compute global P95 for every (source, cmdb_id, kpi_name) using full series (do NOT filter by time)
global_p95_metric = (
    df_metric
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)
global_p95_metric.insert(0, 'source', 'metric')

global_p95_log = (
    df_log
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_p95'})
)
global_p95_log.insert(0, 'source', 'log')

global_p95 = pd.concat([global_p95_metric, global_p95_log], ignore_index=True)

# 2) Filter each file to the incident window (inclusive)
window_metric = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()
window_log    = df_log[(df_log['timestamp']    >= incident_start) & (df_log['timestamp']    <= incident_end)].copy()

# Incident-window row counts for each file
metric_window_count = window_metric.shape[0]
log_window_count = window_log.shape[0]

# 3) Mark anomalies where value >= global_p95 (merge thresholds into windowed data)
# Merge for metric
wm = window_metric.merge(global_p95_metric, on=['cmdb_id', 'kpi_name'], how='left')
wm['anomaly'] = wm['value'] >= wm['global_p95']

# Merge for log
wl = window_log.merge(global_p95_log, on=['cmdb_id', 'kpi_name'], how='left')
wl['anomaly'] = wl['value'] >= wl['global_p95']

# 4) For each (source, cmdb_id, kpi_name) produce required summary
def summarize_window(df_window, source_label):
    # df_window already contains columns: timestamp, cmdb_id, kpi_name, value, global_p95, anomaly
    g = df_window.groupby(['cmdb_id', 'kpi_name'], as_index=False)
    summary = g.agg(
        anomaly_count_in_window = ('anomaly', 'sum'),
        earliest_anomaly_ts = ('timestamp', lambda x: x.loc[df_window.loc[x.index, 'anomaly']].min() if (df_window.loc[x.index, 'anomaly'].any()) else pd.NaT),
        max_value_in_window = ('value', 'max'),
        global_p95 = ('global_p95', 'first')
    )
    summary.insert(0, 'source', source_label)
    # Compute ratio, set to NaN/None if global_p95 == 0 or missing
    def ratio(row):
        gp = row['global_p95']
        mv = row['max_value_in_window']
        if pd.isna(gp) or gp == 0:
            return np.nan
        return mv / gp if not pd.isna(mv) else np.nan
    summary['max_over_p95_ratio'] = summary.apply(ratio, axis=1)
    # Format earliest_anomaly_ts as ISO UTC strings or None
    summary['earliest_anomaly_timestamp'] = summary['earliest_anomaly_ts'].apply(
        lambda ts: ts.isoformat() if pd.notna(ts) else None
    )
    # Keep only requested columns and order
    summary = summary[[
        'source', 'cmdb_id', 'kpi_name', 'global_p95',
        'anomaly_count_in_window', 'earliest_anomaly_timestamp',
        'max_value_in_window', 'max_over_p95_ratio'
    ]]
    return summary

summary_metric = summarize_window(wm, 'metric')
summary_log = summarize_window(wl, 'log')

# Combine summaries and sort as required
anomaly_summary = pd.concat([summary_metric, summary_log], ignore_index=True)

# Replace inf with NaN, and round numeric columns for compactness
anomaly_summary['global_p95'] = pd.to_numeric(anomaly_summary['global_p95'], errors='coerce')
anomaly_summary['max_value_in_window'] = pd.to_numeric(anomaly_summary['max_value_in_window'], errors='coerce')
anomaly_summary['max_over_p95_ratio'] = pd.to_numeric(anomaly_summary['max_over_p95_ratio'], errors='coerce')

anomaly_summary = anomaly_summary.sort_values(
    by=['anomaly_count_in_window', 'max_over_p95_ratio'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Compact numeric formatting: round to sensible precision
for col in ['global_p95', 'max_value_in_window', 'max_over_p95_ratio']:
    if col in anomaly_summary.columns:
        anomaly_summary[col] = anomaly_summary[col].round(6)

# Prepare window counts output
window_counts = pd.Series({
    'metric_window_count': metric_window_count,
    'log_window_count': log_window_count
})

# Final outputs: anomaly_summary (compact, <=20 rows) and window_counts (two numbers)
anomaly_summary, window_counts

```
Out[2]:
```
Summary of anomaly detection (incident window 2024-01-19 15:42:56 → 2024-01-19 16:12:56 UTC):

- Window row counts:
  - metric.csv: 1950 rows in the incident window.
  - log.csv: 450 rows in the incident window.

- Main findings (top 20 results shown):
  - Many metric KPIs (notably socket metrics across most services: carts-db, catalogue, catalogue-db, orders-db, queue-master, rabbitmq, rabbitmq-exporter, session-db, user, user-db) have anomaly_count_in_window = 25 and max_over_p95_ratio = 1.0. Earliest anomaly timestamps for these are 2024-01-19T15:45:00+00:00. This means every sampled point in the incident window met or exceeded the global P95 and the maximum value in-window equals the global P95.
  - Several metric KPIs (e.g., carts.diskio, catalogue-db.diskio, various .error KPIs) and the two log KPIs (catalogue.log.error_count, front-end.log.error_count) have global_p95 == 0 and anomaly_count_in_window = 25. Because global_p95 is zero, max_over_p95_ratio is not computable (shown as null/NaN). These are flagged as anomalies only by the >= global_p95 rule, but the zero threshold makes the ratio meaningless.
  - Across the reported rows, the earliest detected anomaly for every listed KPI is 2024-01-19T15:45:00+00:00.

- Important caveat:
  - Several KPIs have global P95 == 0. Marking values >= 0 as "anomalies" will mark any non-negative sample; therefore those results should be interpreted with caution. The max_over_p95_ratio is undefined when global_p95 == 0.

If you want, I can:
- Expand beyond the top 20 rows,
- Exclude KPIs whose global P95 == 0 (to avoid the zero-threshold noise),
- Or list only KPIs with max_over_p95_ratio > 1 for stronger anomalies.

The original code execution output of IPython Kernel is also provided below for reference:

(    source            cmdb_id         kpi_name  global_p95  anomaly_count_in_window earliest_anomaly_timestamp  max_value_in_window  max_over_p95_ratio
0   metric           carts-db           socket         7.0                       25  2024-01-19T15:45:00+00:00         7.000000e+00                 1.0
1   metric          catalogue           socket         6.0                       25  2024-01-19T15:45:00+00:00         6.000000e+00                 1.0
2   metric       catalogue-db           socket         4.0                       25  2024-01-19T15:45:00+00:00         4.000000e+00                 1.0
3   metric          orders-db           socket         5.0                       25  2024-01-19T15:45:00+00:00         5.000000e+00                 1.0
4   metric       queue-master           socket         3.0                       25  2024-01-19T15:45:00+00:00         3.000000e+00                 1.0
5   metric           rabbitmq           socket        11.0                       25  2024-01-19T15:45:00+00:00         1.100000e+01                 1.0
6   metric  rabbitmq-exporter              mem   4157440.0                       25  2024-01-19T15:45:00+00:00         4.157440e+06                 1.0
7   metric  rabbitmq-exporter           socket         1.0                       25  2024-01-19T15:45:00+00:00         1.000000e+00                 1.0
8   metric         session-db           socket         3.0                       25  2024-01-19T15:45:00+00:00         3.000000e+00                 1.0
9   metric               user           socket        17.0                       25  2024-01-19T15:45:00+00:00         1.700000e+01                 1.0
10  metric            user-db           socket         8.0                       25  2024-01-19T15:45:00+00:00         8.000000e+00                 1.0
11  metric              carts           diskio         0.0                       25  2024-01-19T15:45:00+00:00         1.306356e+05                 NaN
12  metric              carts            error         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
13  metric       catalogue-db           diskio         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
14  metric          front-end            error         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
15  metric             orders            error         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
16  metric           rabbitmq           diskio         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
17  metric               user            error         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
18     log          catalogue  log.error_count         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN
19     log          front-end  log.error_count         0.0                       25  2024-01-19T15:45:00+00:00         0.000000e+00                 NaN, metric_window_count    1950
log_window_count        450
dtype: int64)```
```

In [ ]:
# In[3]:

# Reuse previously computed windowed merged DataFrames wm (metric) and wl (log)
import numpy as np
import pandas as pd

# Helper to summarize groups for strict >1 ratio condition
def summarize_strict(df_window, source_label):
    # consider only rows where global_p95 is present and > 0 to avoid zero-division
    df = df_window.copy()
    # ensure global_p95 column exists
    if 'global_p95' not in df.columns:
        return pd.DataFrame(columns=[
            'source','cmdb_id','kpi_name','global_p95','anomaly_count_in_window',
            'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','max_over_p95_ratio'
        ])
    # Group and compute
    records = []
    for (cmdb, kpi), grp in df.groupby(['cmdb_id', 'kpi_name']):
        gp = grp['global_p95'].iloc[0]
        if pd.isna(gp) or gp <= 0:
            continue
        max_val = grp['value'].max()
        ratio = max_val / gp if gp != 0 else np.nan
        if ratio <= 1.0 or pd.isna(ratio):
            continue
        # anomaly count uses the rule value >= global_p95
        anomaly_mask = grp['value'] >= gp
        anomaly_count = int(anomaly_mask.sum())
        # earliest and latest anomaly timestamps (only from anomaly rows)
        if anomaly_count > 0:
            times = grp.loc[anomaly_mask, 'timestamp']
            earliest = times.min()
            latest = times.max()
            earliest_iso = earliest.isoformat()
            latest_iso = latest.isoformat()
        else:
            earliest_iso = None
            latest_iso = None
        records.append({
            'source': source_label,
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'global_p95': gp,
            'anomaly_count_in_window': anomaly_count,
            'earliest_anomaly_timestamp': earliest_iso,
            'latest_anomaly_timestamp': latest_iso,
            'max_value_in_window': max_val,
            'max_over_p95_ratio': ratio
        })
    return pd.DataFrame.from_records(records)

# Summarize for metric and log
summary_metric_strict = summarize_strict(wm, 'metric')
summary_log_strict = summarize_strict(wl, 'log')

# Combine, sort, limit to top 20
combined_strict = pd.concat([summary_metric_strict, summary_log_strict], ignore_index=True)
combined_strict = combined_strict.sort_values(
    by=['max_over_p95_ratio', 'anomaly_count_in_window'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Format numeric columns for compactness
for col in ['global_p95', 'max_value_in_window', 'max_over_p95_ratio']:
    if col in combined_strict.columns:
        combined_strict[col] = combined_strict[col].round(6)

# Count distinct (source, cmdb_id) components in the filtered set
distinct_components_count = combined_strict[['source','cmdb_id']].drop_duplicates().shape[0]

# Final outputs: compact table and distinct component count
combined_strict, distinct_components_count

```
Out[3]:
```
Summary of KPIs that exceeded their global P95 (global_p95 > 0 and max_value_in_window / global_p95 > 1.0):

- Total matching rows returned: 20 (top by ratio).  
- Distinct (source, cmdb_id) components in this set: 16.

Top matches (by max_over_p95_ratio):
1. metric.orders cpu — global_p95=2.587024, anomalies=2, earliest=2024-01-19T15:47:00+00:00, latest=2024-01-19T15:49:00+00:00, max=5.386233, ratio≈2.0820
2. log.carts-db log.row_count — global_p95=1.0, anomalies=24, earliest=2024-01-19T15:46:00+00:00, latest=2024-01-19T16:09:00+00:00, max=2.0, ratio=2.0
3. metric.rabbitmq-exporter cpu — global_p95=0.004167, anomalies=2, earliest=2024-01-19T15:45:00+00:00, latest=2024-01-19T15:52:00+00:00, max=0.006649, ratio≈1.5956
4. metric.shipping cpu — global_p95=0.563449, anomalies=2, earliest=2024-01-19T16:00:00+00:00, latest=2024-01-19T16:02:00+00:00, max=0.767458, ratio≈1.3621
5. metric.session-db diskio — global_p95=13555.40, anomalies=2, earliest=2024-01-19T15:53:00+00:00, latest=2024-01-19T16:03:00+00:00, max=16981.20, ratio≈1.2527

Other notable entries include log KPIs for queue-master, orders, shipping (log.row_count ratios ≈1.0638), metric queue-master/user-db diskio and several cpu/latency KPIs with ratios between ~1.05–1.18. Most matched KPIs have anomaly_count_in_window = 2; exceptions with many anomalies include carts-db log.row_count (24) and shipping.socket (16).

If you want, I can:
- return the compact table as a dataframe,
- filter to only high-severity cases (e.g., ratio > 1.2), or
- show timestamps and values for specific KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(    source            cmdb_id       kpi_name    global_p95  anomaly_count_in_window earliest_anomaly_timestamp   latest_anomaly_timestamp  max_value_in_window  max_over_p95_ratio
0   metric             orders            cpu  2.587024e+00                        2  2024-01-19T15:47:00+00:00  2024-01-19T15:49:00+00:00         5.386233e+00            2.082019
1      log           carts-db  log.row_count  1.000000e+00                       24  2024-01-19T15:46:00+00:00  2024-01-19T16:09:00+00:00         2.000000e+00            2.000000
2   metric  rabbitmq-exporter            cpu  4.167000e-03                        2  2024-01-19T15:45:00+00:00  2024-01-19T15:52:00+00:00         6.649000e-03            1.595625
3   metric           shipping            cpu  5.634490e-01                        2  2024-01-19T16:00:00+00:00  2024-01-19T16:02:00+00:00         7.674580e-01            1.362071
4   metric         session-db         diskio  1.355540e+04                        2  2024-01-19T15:53:00+00:00  2024-01-19T16:03:00+00:00         1.698120e+04            1.252726
5      log              carts  log.row_count  1.250000e+02                        2  2024-01-19T16:02:00+00:00  2024-01-19T16:08:00+00:00         1.500000e+02            1.200000
6   metric       queue-master            cpu  3.560960e-01                        2  2024-01-19T15:53:00+00:00  2024-01-19T16:02:00+00:00         4.199350e-01            1.179274
7   metric            user-db         diskio  1.732200e+06                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:06:00+00:00         2.012200e+06            1.161644
8   metric       queue-master         diskio  5.679773e+04                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:02:00+00:00         6.564979e+04            1.155852
9   metric              carts     latency-90  6.019100e-02                        2  2024-01-19T16:06:00+00:00  2024-01-19T16:07:00+00:00         6.532600e-02            1.085308
10  metric           carts-db         diskio  2.233593e+06                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:04:00+00:00         2.410765e+06            1.079321
11  metric            user-db            cpu  9.657370e-01                        2  2024-01-19T15:45:00+00:00  2024-01-19T15:46:00+00:00         1.035795e+00            1.072543
12  metric               user     latency-90  6.507000e-03                        2  2024-01-19T15:45:00+00:00  2024-01-19T15:46:00+00:00         6.929000e-03            1.064832
13     log       queue-master  log.row_count  5.922000e+02                        2  2024-01-19T16:02:00+00:00  2024-01-19T16:06:00+00:00         6.300000e+02            1.063830
14     log             orders  log.row_count  1.316000e+02                        2  2024-01-19T16:02:00+00:00  2024-01-19T16:06:00+00:00         1.400000e+02            1.063830
15     log           shipping  log.row_count  6.580000e+01                        2  2024-01-19T16:02:00+00:00  2024-01-19T16:06:00+00:00         7.000000e+01            1.063830
16  metric          front-end            cpu  4.743587e+00                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:00:00+00:00         5.031926e+00            1.060785
17  metric          catalogue            cpu  1.852590e-01                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:04:00+00:00         1.963200e-01            1.059701
18  metric           shipping         socket  6.000000e+00                       16  2024-01-19T15:45:00+00:00  2024-01-19T16:09:00+00:00         6.333333e+00            1.055556
19  metric              carts       workload  8.592147e+00                        2  2024-01-19T15:45:00+00:00  2024-01-19T16:02:00+00:00         9.067000e+00            1.055266, 16)```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Candidates list
candidates = [
    {'source': 'metric', 'cmdb_id': 'orders', 'kpi_name': 'cpu'},
    {'source': 'log',    'cmdb_id': 'carts-db', 'kpi_name': 'log.row_count'},
    {'source': 'metric', 'cmdb_id': 'shipping', 'kpi_name': 'socket'},
    {'source': 'metric', 'cmdb_id': 'queue-master', 'kpi_name': 'diskio'},
    {'source': 'metric', 'cmdb_id': 'user-db', 'kpi_name': 'diskio'},
    {'source': 'metric', 'cmdb_id': 'session-db', 'kpi_name': 'diskio'},
    {'source': 'metric', 'cmdb_id': 'rabbitmq-exporter', 'kpi_name': 'cpu'},
    {'source': 'metric', 'cmdb_id': 'carts', 'kpi_name': 'workload'},
]

# Ensure wm and wl (windowed merged with global_p95 and anomaly) exist; if not, recreate them
# (They were computed previously, but recreate safely from df_metric/df_log + global_p95 Metric/Log)
try:
    wm  # noqa: F821
    wl  # noqa: F821
except NameError:
    # Recreate global p95s if needed
    global_p95_metric = df_metric.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    global_p95_metric.insert(0,'source','metric')
    global_p95_log = df_log.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_p95'})
    global_p95_log.insert(0,'source','log')
    # window bounds (from earlier)
    incident_start = pd.to_datetime('2024-01-19 15:42:56', utc=True)
    incident_end   = pd.to_datetime('2024-01-19 16:12:56', utc=True)
    window_metric = df_metric[(df_metric['timestamp'] >= incident_start) & (df_metric['timestamp'] <= incident_end)].copy()
    window_log    = df_log[(df_log['timestamp']    >= incident_start) & (df_log['timestamp']    <= incident_end)].copy()
    wm = window_metric.merge(global_p95_metric, on=['cmdb_id','kpi_name'], how='left')
    wl = window_log.merge(global_p95_log, on=['cmdb_id','kpi_name'], how='left')
    wm['anomaly'] = wm['value'] >= wm['global_p95']
    wl['anomaly'] = wl['value'] >= wl['global_p95']

runs_list = []
candidate_summaries = []

for c in candidates:
    src = c['source']
    cmdb = c['cmdb_id']
    kpi = c['kpi_name']
    # select appropriate windowed df
    dfw = wm if src == 'metric' else wl
    # Filter to candidate rows
    mask = (dfw['cmdb_id'] == cmdb) & (dfw['kpi_name'] == kpi)
    df_cand = dfw.loc[mask].copy()
    # Select anomaly rows where value >= global_p95 (also ensure global_p95 exists)
    # Note: keep only rows where global_p95 is finite
    df_cand = df_cand[df_cand['global_p95'].notna()]
    df_anom = df_cand.loc[df_cand['value'] >= df_cand['global_p95']].sort_values('timestamp').copy()
    total_anomaly_points = df_anom.shape[0]
    # If no anomaly points, record summary and continue
    if total_anomaly_points == 0:
        candidate_summaries.append({
            'source': src,
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'total_anomaly_points_in_window': 0,
            'number_of_runs': 0
        })
        continue
    # Identify consecutive runs where adjacent timestamps differ by exactly 60 seconds
    ts_diff = df_anom['timestamp'].diff().dt.total_seconds().fillna(0)
    # A new run starts when diff != 60
    run_id = (ts_diff != 60).cumsum()
    df_anom['run_id'] = run_id
    runs = []
    for rid, grp in df_anom.groupby('run_id'):
        run_start = grp['timestamp'].min()
        run_end = grp['timestamp'].max()
        run_len = grp.shape[0]
        max_val = grp['value'].max()
        gp = grp['global_p95'].iloc[0]
        max_over = max_val / gp if (pd.notna(gp) and gp != 0) else np.nan
        runs.append({
            'source': src,
            'cmdb_id': cmdb,
            'kpi_name': kpi,
            'run_start_timestamp': run_start.isoformat(),
            'run_end_timestamp': run_end.isoformat(),
            'run_length_points': run_len,
            'max_value_in_run': max_val,
            'max_over_p95_ratio': max_over
        })
    runs_df = pd.DataFrame.from_records(runs)
    runs_list.append(runs_df)
    candidate_summaries.append({
        'source': src,
        'cmdb_id': cmdb,
        'kpi_name': kpi,
        'total_anomaly_points_in_window': int(total_anomaly_points),
        'number_of_runs': int(runs_df.shape[0])
    })

# Combine all runs into one DataFrame
if runs_list:
    all_runs = pd.concat(runs_list, ignore_index=True)
else:
    all_runs = pd.DataFrame(columns=[
        'source','cmdb_id','kpi_name','run_start_timestamp','run_end_timestamp',
        'run_length_points','max_value_in_run','max_over_p95_ratio'
    ])

# Sort and limit to top 20 runs by run_length_points desc, then max_over_p95_ratio desc
all_runs_sorted = all_runs.sort_values(
    by=['run_length_points', 'max_over_p95_ratio'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# Round numeric columns for compactness
for col in ['max_value_in_run', 'max_over_p95_ratio']:
    if col in all_runs_sorted.columns:
        all_runs_sorted[col] = pd.to_numeric(all_runs_sorted[col]).round(6)

# Candidate summaries DataFrame
candidates_summary_df = pd.DataFrame.from_records(candidate_summaries)

# Final outputs (compact): all_runs_sorted (<=20 rows) and candidates_summary_df
all_runs_sorted, candidates_summary_df

```
Out[4]:
```
Summary of fault-identification for the eight candidate KPIs (incident window 2024-01-19 15:42:56 → 2024-01-19 16:12:56 UTC):

Per-candidate totals (total anomaly points in window, number of consecutive runs):
- metric.orders cpu: 2 anomaly points, 2 runs.
- log.carts-db log.row_count: 24 anomaly points, 1 run.
- metric.shipping socket: 16 anomaly points, 6 runs.
- metric.queue-master diskio: 2 anomaly points, 2 runs.
- metric.user-db diskio: 2 anomaly points, 2 runs.
- metric.session-db diskio: 2 anomaly points, 2 runs.
- metric.rabbitmq-exporter cpu: 2 anomaly points, 2 runs.
- metric.carts workload: 2 anomaly points, 2 runs.

Top runs (sorted by run length then ratio) — main findings:
1. log.carts-db / log.row_count — run 2024-01-19T15:46:00→2024-01-19T16:09:00 (24 points), max=2.0, max/global_p95 ratio=2.00. This is a long, sustained anomalous run.
2. metric.shipping / socket — run 2024-01-19T15:45:00→2024-01-19T15:51:00 (7 points), max=6.0, ratio=1.00 (meets P95).
3. metric.shipping / socket — run 2024-01-19T16:04:00→2024-01-19T16:06:00 (3 points), max=6.333333, ratio≈1.0556.
4. metric.shipping / socket — run 2024-01-19T15:53:00→2024-01-19T15:55:00 (3 points), max=6.0, ratio=1.00.
5. metric.orders / cpu — two single-point runs:
   - 2024-01-19T15:47:00 (1 point), max=5.386233, ratio≈2.0820 (highest severity for orders.cpu).
   - 2024-01-19T15:49:00 (1 point), max=2.729675, ratio≈1.0551.
6. metric.rabbitmq-exporter / cpu — two single-point runs (15:45 and 15:52), max ratio ≈1.5956 at 15:45.
7. metric.session-db / diskio — two single-point runs; the 16:03 run has max=16981.20, ratio≈1.2527.
8. metric.user-db / diskio and metric.queue-master / diskio — each have two single-point runs with ratios ≈1.16 and ≈1.16/1.04 respectively.
9. metric.carts / workload — two single-point runs with modest ratios (~1.055 and ~1.0036).

Interpretation / actionable points:
- The most notable sustained anomaly is log.carts-db log.row_count (24 consecutive minutes, ratio 2.0). Investigate logs/ingestion and carts-db behavior.
- shipping.socket shows several short consecutive runs totaling 16 anomaly points; one short run (3 points) exceeds P95 slightly (ratio ≈1.0556). Check shipping socket behavior and connectivity around those times.
- orders.cpu had brief but high-severity spikes (one spike with ratio ≈2.08). Investigate orders service CPU spikes at 15:47 and 15:49.
- session-db/user-db/queue-master diskio anomalies are single-point spikes with ratios >1 — investigate I/O bursts on those DB/queue components.
- rabbitmq-exporter cpu shows small but non-trivial spikes; check exporter/process CPU around 15:45 and 15:52.

If you want, I can:
- show the exact anomaly timestamps for any single candidate,
- filter runs to only multi-point runs or only runs with ratio > 1.2.

The original code execution output of IPython Kernel is also provided below for reference:

(    source            cmdb_id       kpi_name        run_start_timestamp          run_end_timestamp  run_length_points  max_value_in_run  max_over_p95_ratio
0      log           carts-db  log.row_count  2024-01-19T15:46:00+00:00  2024-01-19T16:09:00+00:00                 24      2.000000e+00            2.000000
1   metric           shipping         socket  2024-01-19T15:45:00+00:00  2024-01-19T15:51:00+00:00                  7      6.000000e+00            1.000000
2   metric           shipping         socket  2024-01-19T16:04:00+00:00  2024-01-19T16:06:00+00:00                  3      6.333333e+00            1.055556
3   metric           shipping         socket  2024-01-19T15:53:00+00:00  2024-01-19T15:55:00+00:00                  3      6.000000e+00            1.000000
4   metric             orders            cpu  2024-01-19T15:47:00+00:00  2024-01-19T15:47:00+00:00                  1      5.386233e+00            2.082019
5   metric  rabbitmq-exporter            cpu  2024-01-19T15:45:00+00:00  2024-01-19T15:45:00+00:00                  1      6.649000e-03            1.595625
6   metric         session-db         diskio  2024-01-19T16:03:00+00:00  2024-01-19T16:03:00+00:00                  1      1.698120e+04            1.252726
7   metric            user-db         diskio  2024-01-19T15:45:00+00:00  2024-01-19T15:45:00+00:00                  1      2.012200e+06            1.161644
8   metric       queue-master         diskio  2024-01-19T15:45:00+00:00  2024-01-19T15:45:00+00:00                  1      6.564979e+04            1.155852
9   metric              carts       workload  2024-01-19T15:45:00+00:00  2024-01-19T15:45:00+00:00                  1      9.067000e+00            1.055266
10  metric             orders            cpu  2024-01-19T15:49:00+00:00  2024-01-19T15:49:00+00:00                  1      2.729675e+00            1.055141
11  metric       queue-master         diskio  2024-01-19T16:02:00+00:00  2024-01-19T16:02:00+00:00                  1      5.907766e+04            1.040141
12  metric         session-db         diskio  2024-01-19T15:53:00+00:00  2024-01-19T15:53:00+00:00                  1      1.375924e+04            1.015038
13  metric  rabbitmq-exporter            cpu  2024-01-19T15:52:00+00:00  2024-01-19T15:52:00+00:00                  1      4.229000e-03            1.014920
14  metric              carts       workload  2024-01-19T16:02:00+00:00  2024-01-19T16:02:00+00:00                  1      8.623250e+00            1.003620
15  metric            user-db         diskio  2024-01-19T16:06:00+00:00  2024-01-19T16:06:00+00:00                  1      1.735229e+06            1.001749
16  metric           shipping         socket  2024-01-19T15:58:00+00:00  2024-01-19T15:58:00+00:00                  1      6.000000e+00            1.000000
17  metric           shipping         socket  2024-01-19T16:02:00+00:00  2024-01-19T16:02:00+00:00                  1      6.000000e+00            1.000000
18  metric           shipping         socket  2024-01-19T16:09:00+00:00  2024-01-19T16:09:00+00:00                  1      6.000000e+00            1.000000,    source            cmdb_id       kpi_name  total_anomaly_points_in_window  number_of_runs
0  metric             orders            cpu                               2               2
1     log           carts-db  log.row_count                              24               1
2  metric           shipping         socket                              16               6
3  metric       queue-master         diskio                               2               2
4  metric            user-db         diskio                               2               2
5  metric         session-db         diskio                               2               2
6  metric  rabbitmq-exporter            cpu                               2               2
7  metric              carts       workload                               2               2)```
```